In [ ]:
from pathlib import Path
from typing import List

import pymupdf4llm
from src.models.document import Document, DocumentMetadata

def load_pdf(file_path: Path) -> List[Document]:
    """
    Parses multi-column PDFs into clean markdown chunks per page 
    using layout heuristics. Zero GPU required.
    """
    pdf_path = Path(file_path)
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF file not found: {pdf_path}")

    # page_chunks=True returns a list of dictionaries, one per page
    pages_data = pymupdf4llm.to_markdown(str(pdf_path), page_chunks=True)
    documents: List[Document] = []

    for page in pages_data:
        text = page["text"]
        page_num = page["metadata"]["page_number"]
        
        if not text.strip():
            continue

        metadata = DocumentMetadata(
            source=pdf_path.name,
            file_type="pdf",
            page=page_num
        )
        
        documents.append(Document(page_content=text, metadata=metadata))
        
    return documents

In [ ]:
docs = load_pdf(file_path="../documents/pdfs/MacBook Air (13-inch, M5) - Tech Specs.pdf") 

In [4]:
docs

[Document(page_content='5/28/26, 11:20 AM \n\nMacBook Air (13-inch, M5) - Tech Specs - Apple Support (IN) \n\n**Documentation** \n\n**==> picture [99 x 99] intentionally omitted <==**\n\n## **MacBook Air (13-inch, M5) - Tech Specs** \n\nYear introduced: 2026 \n\n## **Finish** \n\nSky Blue Silver Starlight Midnight \n\n## **Chip** \n\n## **Apple M5 chip** \n\n10-core CPU with 4 super cores and 6 efficiency cores \n\n8-core GPU, 10-core GPU Neural Accelerators Hardware-accelerated ray tracing 16-core Neural Engine 153GB/s memory bandwidth \n\n## **Media Engine** \n\nHardware-accelerated H.264, HEVC, ProRes and ProRes RAW \n\nVideo decode engine Video encode engine ProRes encode and decode engine AV1 decode \n\n## **Configurable to:** \n\nM5 with 10-core CPU and 10-core GPU \n\nhttps://support.apple.com/en-in/126320 \n\n1/7 \n\n', metadata=DocumentMetadata(source='MacBook Air (13-inch, M5) - Tech Specs.pdf', file_type='pdf', page=1, section=None, chunk_id=None, chunk_index=None, parent_do

In [17]:
docs[0].page_content

'5/28/26, 11:20 AM \n\nMacBook Air (13-inch, M5) - Tech Specs - Apple Support (IN) \n\n**Documentation** \n\n**==> picture [99 x 99] intentionally omitted <==**\n\n## **MacBook Air (13-inch, M5) - Tech Specs** \n\nYear introduced: 2026 \n\n## **Finish** \n\nSky Blue Silver Starlight Midnight \n\n## **Chip** \n\n## **Apple M5 chip** \n\n10-core CPU with 4 super cores and 6 efficiency cores \n\n8-core GPU, 10-core GPU Neural Accelerators Hardware-accelerated ray tracing 16-core Neural Engine 153GB/s memory bandwidth \n\n## **Media Engine** \n\nHardware-accelerated H.264, HEVC, ProRes and ProRes RAW \n\nVideo decode engine Video encode engine ProRes encode and decode engine AV1 decode \n\n## **Configurable to:** \n\nM5 with 10-core CPU and 10-core GPU \n\nhttps://support.apple.com/en-in/126320 \n\n1/7 \n\n'

# Cleaning / Preprocessing

In [5]:
import re
from typing import List

from src.models.document import Document

class PDFTextCleaner: 
    def __init__(self): 
        pass

    def remove_headers_and_footers(self, text: str) -> str:
        """
        Remove repeated PDF header/footer noise.
        """

        # Remove timestamp lines
        text = re.sub(
            r"\d{1,2}/\d{1,2}/\d{2},\s+\d{1,2}:\d{2}\s+[AP]M",
            "",
            text
        )

        # Remove Apple Support page title lines
        text = re.sub(
            r"MacBook Air \(13-inch, M5\) - Tech Specs - Apple Support \(IN\)",
            "",
            text
        )

        # Remove URLs
        text = re.sub(
            r"https?://\S+",
            "",
            text
        )

        # Remove page counters like 1/7
        text = re.sub(
            r"\b\d+/\d+\b",
            "",
            text
        )

        return text


    def remove_image_placeholders(self, text: str) -> str:
        """
        Remove pymupdf4llm image placeholder text.
        """

        text = re.sub(
            r"\*\*==> picture.*?omitted <==\*\*",
            "",
            text
        )

        return text


    def fix_broken_words(self, text: str) -> str:
        """
        Fix common OCR-style broken words conservatively.
        """

        replacements = {
            "Con fig ure": "Configure",
            "En viron men tal": "Environmental",
            "Accessi bility": "Accessibility",
        }

        for broken, fixed in replacements.items():
            text = text.replace(broken, fixed)

        return text


    def normalize_whitespace(self, text: str) -> str:
        """
        Normalize whitespace while preserving markdown structure.
        """

        # Remove trailing spaces
        text = re.sub(r"[ \t]+$", "", text, flags=re.MULTILINE)

        # Replace excessive blank lines (3+ → 2)
        text = re.sub(r"\n{3,}", "\n\n", text)

        # Strip leading/trailing whitespace
        text = text.strip()

        return text
    
   
    def remove_footer_noise(self, text: str) -> str:
        """
        Remove remaining website footer artifacts specifically at 7th page.
        """

        footer_patterns = [
            r"\*\*Helpful\?\*\* Yes No",
            r"Support MacBook Air.*",
            r"Copyright © .*",
            r"Privacy Policy",
            r"Terms of Use",
            r"Sales Policy",
            r"Site Map",
            r"\bIndia\b",
        ]

        for pattern in footer_patterns:
            text = re.sub(pattern, "", text)

        return text


    def fix_footnote_artifacts(self, text: str) -> str:
        """
        Fix merged footnote/superscript artifacts conservatively.
        Examples:
            Storage1 -> Storage
            Power3 -> Power
        """

        text = re.sub(
            r"\b([A-Za-z]{5,})(\d{1,2})\b",
            r"\1",
            text
        )

        return text




    def clean_text(self, text: str) -> str:
        """
        Apply full text cleaning pipeline.
        """

        text = self.remove_headers_and_footers(text)
        text = self.remove_image_placeholders(text)
        text = self.fix_broken_words(text)
        text = self.normalize_whitespace(text)
        text = self.remove_footer_noise(text)
        text = self.fix_footnote_artifacts(text)

        return text


    def clean_documents(self, documents: List[Document]) -> List[Document]:
        """
        Clean a list of Document objects while preserving metadata.
        """

        cleaned_documents: List[Document] = []

        for document in documents:
            cleaned_content = self.clean_text(document.page_content)

            cleaned_document = Document(
                page_content=cleaned_content,
                metadata=document.metadata
            )

            cleaned_documents.append(cleaned_document)

        return cleaned_documents



In [6]:
cleaner = PDFTextCleaner()
pre_docs = cleaner.clean_documents(docs)

In [7]:
pre_docs

[Document(page_content='**Documentation**\n\n## **MacBook Air (13-inch, M5) - Tech Specs**\n\nYear introduced: 2026\n\n## **Finish**\n\nSky Blue Silver Starlight Midnight\n\n## **Chip**\n\n## **Apple M5 chip**\n\n10-core CPU with 4 super cores and 6 efficiency cores\n\n8-core GPU, 10-core GPU Neural Accelerators Hardware-accelerated ray tracing 16-core Neural Engine 153GB/s memory bandwidth\n\n## **Media Engine**\n\nHardware-accelerated H.264, HEVC, ProRes and ProRes RAW\n\nVideo decode engine Video encode engine ProRes encode and decode engine AV1 decode\n\n## **Configurable to:**\n\nM5 with 10-core CPU and 10-core GPU', metadata=DocumentMetadata(source='MacBook Air (13-inch, M5) - Tech Specs.pdf', file_type='pdf', page=1, section=None, chunk_id=None, chunk_index=None, parent_document_id=None)),
 Document(page_content='**Memory** 16GB unified memory **Configurable to:** 24GB or 32GB **Storage** 512GB SSD **Configurable to:** 1TB, 2TB or 4TB **Display Liquid Retina display** 13.6-inch 

# Chunking

In [ ]:
from pathlib import Path
from typing import List

from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter

from src.models.document import Document, DocumentMetadata


# =========================================================
# SPLITTER CONFIGURATION
# =========================================================

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "header_1"),
        ("##", "header_2"),
        ("###", "header_3"),
    ], 
    strip_headers=False 
)


# =========================================================
# HELPER FUNCTIONS
# =========================================================

def extract_section_name(metadata: dict) -> str | None:
    """
    Extract the most specific available markdown header.
    """

    return (
        metadata.get("header_3")
        or metadata.get("header_2")
        or metadata.get("header_1")
    )


def generate_chunk_id(
    source: str,
    chunk_index: int,
    page: int | None = None
) -> str:
    """
    Generate deterministic readable chunk IDs.
    """

    stem = Path(source).stem.lower().replace(" ", "_")

    if page is not None:
        return f"{stem}_page_{page}_chunk_{chunk_index}"

    return f"{stem}_chunk_{chunk_index}"


def create_chunk_document(
    content: str,
    original_metadata: DocumentMetadata,
    chunk_index: int,
    section: str | None = None,
) -> Document:
    """
    Create a chunked Document object with enriched metadata.
    """

    chunk_id = generate_chunk_id(
        source=original_metadata.source,
        page=original_metadata.page,
        chunk_index=chunk_index,
    )

    parent_document_id = Path(
        original_metadata.source
    ).stem.lower().replace(" ", "_")

    metadata = DocumentMetadata(
        source=original_metadata.source,
        file_type=original_metadata.file_type,
        page=original_metadata.page,
        section=section,
        chunk_id=chunk_id,
        chunk_index=chunk_index,
        parent_document_id=parent_document_id,
    )

    return Document(
        page_content=content,
        metadata=metadata,
    )


# =========================================================
# PDF / MARKDOWN CHUNKING
# =========================================================

def chunk_structured_document(document: Document) -> List[Document]:
    """
    Chunk markdown-structured documents (PDF/MD).
    """

    final_chunks = []

    # Step 1: Split by markdown headers
    header_splits = markdown_splitter.split_text(
        document.page_content
    )

    chunk_counter = 0

    for split in header_splits:

        section = extract_section_name(split.metadata)

        # Step 2: Recursively split oversized sections
        recursive_chunks = recursive_splitter.split_text(
            split.page_content
        )

        for chunk_text in recursive_chunks:

            chunk_doc = create_chunk_document(
                content=chunk_text,
                original_metadata=document.metadata,
                chunk_index=chunk_counter,
                section=section,
            )

            final_chunks.append(chunk_doc)

            chunk_counter += 1

    return final_chunks


# =========================================================
# TXT CHUNKING
# =========================================================

def chunk_text_document(document: Document) -> List[Document]:
    """
    Chunk plain text documents.
    """

    final_chunks = []

    chunks = recursive_splitter.split_text(
        document.page_content
    )

    for idx, chunk_text in enumerate(chunks):

        chunk_doc = create_chunk_document(
            content=chunk_text,
            original_metadata=document.metadata,
            chunk_index=idx,
            section=None,
        )

        final_chunks.append(chunk_doc)

    return final_chunks


# =========================================================
# MAIN DISPATCHER
# =========================================================

def chunk_documents(
    documents: List[Document]
) -> List[Document]:
    """
    Main chunking dispatcher.
    """

    all_chunks = []

    for document in documents:

        file_type = document.metadata.file_type.lower()

        # PDF + Markdown
        if file_type in ["pdf", "md", "markdown"]:

            chunks = chunk_structured_document(document)

        # TXT
        elif file_type == "txt":

            chunks = chunk_text_document(
                document
            )

        else:
            raise ValueError(
                f"Unsupported file type: {file_type}"
            )

        all_chunks.extend(chunks)

    return all_chunks



In [43]:
chunk_docs = chunk_documents(pre_docs)

In [95]:
chunk_docs?

Type:        list
String form: [Document(page_content='**Documentation**', metadata=DocumentMetadata(source='MacBook Air (13-inc <...> cs_page_7_chunk_8', chunk_index=8, parent_document_id='macbook_air_(13-inch,_m5)_-_tech_specs'))]
Length:      41
Docstring:  
Built-in mutable sequence.

If no argument is given, the constructor creates a new empty list.
The argument must be an iterable if specified.

In [98]:
with open('chunk_docs.txt', 'w', encoding='utf-8') as f:
    for item in chunk_docs:
        f.write(f"{item}\n")

In [36]:
chunk_docs[6]

Document(page_content='**Memory** 16GB unified memory **Configurable to:** 24GB or 32GB **Storage** 512GB SSD **Configurable to:** 1TB, 2TB or 4TB **Display Liquid Retina display** 13.6-inch (diagonal) LED-backlit display with IPS technology;2 2560x1664 native resolution at 224 pixels per inch 500 nits brightness **Colour** Support for 1 billion colours Wide colour (P3) True Tone technology **Battery and** Up to 18 hours video streaming **Power** Up to 15 hours wireless web Built-in 53.8-watt-hour lithium-polymer', metadata=DocumentMetadata(source='MacBook Air (13-inch, M5) - Tech Specs.pdf', file_type='pdf', page=2, section=None, chunk_id='macbook_air_(13-inch,_m5)_-_tech_specs_page_2_chunk_0', chunk_index=0, parent_document_id='macbook_air_(13-inch,_m5)_-_tech_specs'))

In [90]:
chunk_indicies = [doc.metadata.chunk_index for doc in chunk_docs]
chunk_indicies

[0,
 1,
 2,
 3,
 4,
 5,
 0,
 1,
 2,
 3,
 0,
 1,
 2,
 3,
 4,
 5,
 0,
 1,
 2,
 3,
 4,
 0,
 1,
 2,
 3,
 4,
 0,
 1,
 2,
 3,
 4,
 5,
 0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8]

In [91]:
chunk_sections = [doc.metadata.section for doc in chunk_docs]
chunk_sections

[None,
 '**MacBook Air (13-inch, M5) - Tech Specs**',
 '**Finish**',
 '**Apple M5 chip**',
 '**Media Engine**',
 '**Configurable to:**',
 None,
 None,
 None,
 None,
 None,
 '**Audio Playback**',
 '**Audio Playback**',
 '**Audio Playback**',
 '**Audio Playback**',
 '**Operating Requirements**',
 None,
 '**Size and Weight**',
 '**macOS**',
 '**Accessibility**',
 '**Features include:**',
 '**In the Box**',
 '**MacBook Air and the Environment**',
 '**Progress towards Apple 2030**',
 '**Materials**',
 '**Energy**',
 None,
 '**Waste**',
 '**Smarter chemistry**',
 '**Acoustic Performance**',
 '**Acoustic Performance**',
 '**Acoustic Performance**',
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

# New Chunking

In [102]:
from pathlib import Path
from typing import List

from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from src.models.document import Document, DocumentMetadata

# =========================================================
# SPLITTER CONFIGURATION
# =========================================================

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "header_1"),
        ("##", "header_2"),
        ("###", "header_3"),
    ], 
    strip_headers=False 
)

# =========================================================
# HELPER FUNCTIONS
# =========================================================

def extract_section_name(metadata: dict) -> str | None:
    """Extract the most specific available markdown header."""
    return (
        metadata.get("header_3")
        or metadata.get("header_2")
        or metadata.get("header_1")
    )


def generate_chunk_id(source: str, chunk_index: int, page: int | None = None) -> str:
    """Generate deterministic, globally unique readable chunk IDs."""
    stem = Path(source).stem.lower().replace(" ", "_")
    if page is not None:
        return f"{stem}_page_{page}_chunk_{chunk_index}"
    return f"{stem}_chunk_{chunk_index}"


def create_chunk_document(
    content: str,
    original_metadata: DocumentMetadata,
    chunk_index: int,
    section: str | None = None,
) -> Document:
    """Create a chunked Document object with enriched metadata."""
    chunk_id = generate_chunk_id(
        source=original_metadata.source,
        page=original_metadata.page,
        chunk_index=chunk_index,
    )

    parent_document_id = Path(original_metadata.source).stem.lower().replace(" ", "_")

    metadata = DocumentMetadata(
        source=original_metadata.source,
        file_type=original_metadata.file_type,
        page=original_metadata.page,
        section=section,
        chunk_id=chunk_id,
        chunk_index=chunk_index,
        parent_document_id=parent_document_id,
    )

    return Document(page_content=content, metadata=metadata)

# =========================================================
# FIXED MAIN DISPATCHER & STRUCTURAL LOOPS
# =========================================================

def chunk_documents_new(documents: List[Document]) -> List[Document]:
    """
    Main chunking dispatcher. Keeps global counters and carries over context state 
    to fix repetitive indexing and null sections across pages.
    """
    all_chunks = []
    
    # FIX 1: Track separate global counters per unique document file
    document_counters = {}
    
    # FIX 2: Track the last running active section layout across page sets
    last_active_section = None

    for document in documents:
        file_type = document.metadata.file_type.lower()
        source_file = document.metadata.source
        
        # Initialize unique file index trackers dynamically
        if source_file not in document_counters:
            document_counters[source_file] = 0

        # --- PROCESS STRUCTURAL MARKDOWN/PDF DOCUMENTS ---
        if file_type in ["pdf", "md", "markdown"]:
            header_splits = markdown_splitter.split_text(document.page_content)

            for split in header_splits:
                extracted_section = extract_section_name(split.metadata)
                
                # Context Carry-Over Strategy: 
                # If LangChain didn't parse a header on this structural slice,
                # use the ongoing last known parent section layout.
                if extracted_section:
                    last_active_section = extracted_section
                
                # Split content into chunk text strings safely
                recursive_chunks = recursive_splitter.split_text(split.page_content)

                for chunk_text in recursive_chunks:
                    chunk_doc = create_chunk_document(
                        content=chunk_text,
                        original_metadata=document.metadata,
                        chunk_index=document_counters[source_file], # Using continuous counter
                        section=last_active_section,               # Restored parent section
                    )
                    all_chunks.append(chunk_doc)
                    document_counters[source_file] += 1

        # --- PROCESS PLAIN TEXT CHUNKS ---
        elif file_type == "txt":
            chunks = recursive_splitter.split_text(document.page_content)

            for chunk_text in chunks:
                chunk_doc = create_chunk_document(
                    content=chunk_text,
                    original_metadata=document.metadata,
                    chunk_index=document_counters[source_file],
                    section=None,
                )
                all_chunks.append(chunk_doc)
                document_counters[source_file] += 1

        else:
            raise ValueError(f"Unsupported file type: {file_type}")

    return all_chunks

In [103]:
chunk_docs_new = chunk_documents_new(pre_docs)

In [104]:
chunk_docs_new?

Type:        list
String form: [Document(page_content='**Documentation**', metadata=DocumentMetadata(source='MacBook Air (13-inc <...> _page_7_chunk_51', chunk_index=51, parent_document_id='macbook_air_(13-inch,_m5)_-_tech_specs'))]
Length:      52
Docstring:  
Built-in mutable sequence.

If no argument is given, the constructor creates a new empty list.
The argument must be an iterable if specified.

In [107]:
chunk_docs_new[:10]

[Document(page_content='**Documentation**', metadata=DocumentMetadata(source='MacBook Air (13-inch, M5) - Tech Specs.pdf', file_type='pdf', page=1, section=None, chunk_id='macbook_air_(13-inch,_m5)_-_tech_specs_page_1_chunk_0', chunk_index=0, parent_document_id='macbook_air_(13-inch,_m5)_-_tech_specs')),
 Document(page_content='## **MacBook Air (13-inch, M5) - Tech Specs**  \nYear introduced: 2026', metadata=DocumentMetadata(source='MacBook Air (13-inch, M5) - Tech Specs.pdf', file_type='pdf', page=1, section='**MacBook Air (13-inch, M5) - Tech Specs**', chunk_id='macbook_air_(13-inch,_m5)_-_tech_specs_page_1_chunk_1', chunk_index=1, parent_document_id='macbook_air_(13-inch,_m5)_-_tech_specs')),
 Document(page_content='## **Finish**  \nSky Blue Silver Starlight Midnight', metadata=DocumentMetadata(source='MacBook Air (13-inch, M5) - Tech Specs.pdf', file_type='pdf', page=1, section='**Finish**', chunk_id='macbook_air_(13-inch,_m5)_-_tech_specs_page_1_chunk_2', chunk_index=2, parent_doc

In [161]:
chunk_docs_new_source = [doc.metadata.source for doc in chunk_docs_new]
chunk_docs_new_source[:10]

['MacBook Air (13-inch, M5) - Tech Specs.pdf',
 'MacBook Air (13-inch, M5) - Tech Specs.pdf',
 'MacBook Air (13-inch, M5) - Tech Specs.pdf',
 'MacBook Air (13-inch, M5) - Tech Specs.pdf',
 'MacBook Air (13-inch, M5) - Tech Specs.pdf',
 'MacBook Air (13-inch, M5) - Tech Specs.pdf',
 'MacBook Air (13-inch, M5) - Tech Specs.pdf',
 'MacBook Air (13-inch, M5) - Tech Specs.pdf',
 'MacBook Air (13-inch, M5) - Tech Specs.pdf',
 'MacBook Air (13-inch, M5) - Tech Specs.pdf']

In [163]:
chunk_docs_new_page = [doc.metadata.page for doc in chunk_docs_new]
chunk_docs_new_page[:10]

[1, 1, 1, 1, 1, 1, 1, 2, 2, 2]

In [164]:
chunk_docs_new_chunk_id = [doc.metadata.chunk_id for doc in chunk_docs_new]
chunk_docs_new_chunk_id[:10]

['macbook_air_(13-inch,_m5)_-_tech_specs_page_1_chunk_0',
 'macbook_air_(13-inch,_m5)_-_tech_specs_page_1_chunk_1',
 'macbook_air_(13-inch,_m5)_-_tech_specs_page_1_chunk_2',
 'macbook_air_(13-inch,_m5)_-_tech_specs_page_1_chunk_3',
 'macbook_air_(13-inch,_m5)_-_tech_specs_page_1_chunk_4',
 'macbook_air_(13-inch,_m5)_-_tech_specs_page_1_chunk_5',
 'macbook_air_(13-inch,_m5)_-_tech_specs_page_1_chunk_6',
 'macbook_air_(13-inch,_m5)_-_tech_specs_page_2_chunk_7',
 'macbook_air_(13-inch,_m5)_-_tech_specs_page_2_chunk_8',
 'macbook_air_(13-inch,_m5)_-_tech_specs_page_2_chunk_9']

In [112]:
chunk_docs_new_indicies = [doc.metadata.chunk_index for doc in chunk_docs_new]
chunk_docs_new_indicies[:30]

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29]

In [114]:
chunk_docs_new_sections = [doc.metadata.section for doc in chunk_docs_new]
chunk_docs_new_sections[:30]

[None,
 '**MacBook Air (13-inch, M5) - Tech Specs**',
 '**Finish**',
 '**Chip**',
 '**Apple M5 chip**',
 '**Media Engine**',
 '**Configurable to:**',
 '**Configurable to:**',
 '**Configurable to:**',
 '**Configurable to:**',
 '**Configurable to:**',
 '**Configurable to:**',
 '**Video Playback**',
 '**Audio Playback**',
 '**Audio Playback**',
 '**Audio Playback**',
 '**Audio Playback**',
 '**Operating Requirements**',
 '**Operating Requirements**',
 '**Size and Weight**',
 '**Operating System**',
 '**macOS**',
 '**macOS**',
 'Learn more about Apple Intelligence',
 '**Accessibility**',
 '**Features include:**',
 '**Features include:**',
 '**In the Box**',
 '**Power adapter sold separately**',
 '**Configure to Order**']

In [166]:
chunk_docs_new_parent_document_id = [doc.metadata.parent_document_id for doc in chunk_docs_new]
chunk_docs_new_parent_document_id[40:50]

['macbook_air_(13-inch,_m5)_-_tech_specs',
 'macbook_air_(13-inch,_m5)_-_tech_specs',
 'macbook_air_(13-inch,_m5)_-_tech_specs',
 'macbook_air_(13-inch,_m5)_-_tech_specs',
 'macbook_air_(13-inch,_m5)_-_tech_specs',
 'macbook_air_(13-inch,_m5)_-_tech_specs',
 'macbook_air_(13-inch,_m5)_-_tech_specs',
 'macbook_air_(13-inch,_m5)_-_tech_specs',
 'macbook_air_(13-inch,_m5)_-_tech_specs',
 'macbook_air_(13-inch,_m5)_-_tech_specs']

# Embeddings

In [140]:
import os
from typing import List, Dict
from dataclasses import asdict
from dotenv import load_dotenv

from google import genai
from google.genai import types

class GeminiEmbeddingGenerator:
    def __init__(self, model_name: str = "gemini-embedding-001"):
        load_dotenv()

        api_key = os.getenv("GEMINI_API_KEY")

        if not api_key:
            raise ValueError("GEMINI_API_KEY not found in environment.")

        self.client = genai.Client(api_key=api_key)

        self.model_name = model_name

    def embed_text(self, text: str) -> List[float]:
        response = self.client.models.embed_content(
            model=self.model_name,
            contents=text,
            config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")
        )

        return response.embedding.values

    def embed_query(self, query: str) -> List[float]:
        response = self.client.models.embed_content(
            model=self.model_name,
            contents=query,
            config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY")
        )

        if hasattr(response, 'embedding') and response.embedding:
            return response.embedding.values
    
        return response.embeddings[0].values

    def embed_documents(self, chunks) -> List[Dict]:

        # Optimization: Extract all text blocks to perform a combined batch request
        texts_to_embed = [chunk.page_content for chunk in chunks]

        response = self.client.models.embed_content(
            model=self.model_name,
            contents=texts_to_embed,
            config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")
        )

        embedded_docs = []

        # When sending a list of texts, response.embeddings contains an iterable array list of records
        for idx, chunk in enumerate(chunks):
            vector = response.embeddings[idx].values

            embedded_docs.append({
                "id": chunk.metadata.chunk_id,
                "text": chunk.page_content,
                "embedding": vector,
                "metadata": asdict(chunk.metadata)
            })

        return embedded_docs

In [115]:
embedder = GeminiEmbeddingGenerator()
embedded_docs = embedder.embed_documents(chunk_docs_new)

In [169]:
embedded_docs[3]['metadata']

{'source': 'MacBook Air (13-inch, M5) - Tech Specs.pdf',
 'file_type': 'pdf',
 'page': 1,
 'section': '**Chip**',
 'chunk_id': 'macbook_air_(13-inch,_m5)_-_tech_specs_page_1_chunk_3',
 'chunk_index': 3,
 'parent_document_id': 'macbook_air_(13-inch,_m5)_-_tech_specs'}

In [117]:
chunk_indices = [doc["metadata"]["chunk_index"] for doc in embedded_docs]

In [118]:
chunk_indices

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51]

In [119]:
print(f"Embedding dimension: {len(embedded_docs[3]['embedding'])}")

Embedding dimension: 3072


In [120]:
embedded_docs?

Type:        list
String form: [{'id': 'macbook_air_(13-inch,_m5)_-_tech_specs_page_1_chunk_0', 'text': '**Documentation**', 'em <...> 7_chunk_51', 'chunk_index': 51, 'parent_document_id': 'macbook_air_(13-inch,_m5)_-_tech_specs'}}]
Length:      52
Docstring:  
Built-in mutable sequence.

If no argument is given, the constructor creates a new empty list.
The argument must be an iterable if specified.

In [167]:
embedded_docs[2]

{'id': 'macbook_air_(13-inch,_m5)_-_tech_specs_page_1_chunk_2',
 'text': '## **Finish**  \nSky Blue Silver Starlight Midnight',
 'embedding': [-0.016151456,
  0.0022815631,
  0.028070139,
  -0.09542838,
  0.015907522,
  0.018714879,
  0.0086561395,
  0.0038753792,
  -0.00510242,
  0.0025982533,
  -0.024500096,
  0.0015422674,
  0.0031464554,
  0.0027893623,
  0.13552184,
  -0.031628817,
  -0.007315899,
  -0.008365996,
  -0.016311413,
  -0.0029118112,
  -0.002835748,
  0.0035172869,
  0.015269494,
  -0.014245555,
  0.006726893,
  -0.024974152,
  0.016080804,
  0.014415702,
  0.05579728,
  -0.0029829773,
  0.001445246,
  0.0013386966,
  0.020716958,
  0.013482115,
  -0.006637006,
  0.035598006,
  0.016792202,
  -0.0062862383,
  0.0072072702,
  0.019045739,
  -0.0029970875,
  0.008688887,
  0.00919395,
  -0.0009987046,
  -0.014099363,
  0.012844336,
  -0.006247525,
  -0.018226583,
  -0.011849599,
  0.017386414,
  -0.021379339,
  0.011392039,
  -0.002663536,
  -0.22321793,
  -0.019414723,


# Vector Database

In [177]:
import os
from typing import List, Dict, Optional, Any, Union
from dotenv import load_dotenv

import psycopg
from pgvector.psycopg import register_vector


class VectorStore:

    def __init__(self):
        load_dotenv()
        self.database_url = os.getenv("DATABASE_URL")

        if not self.database_url:
            raise ValueError("DATABASE_URL not found.")

    def _get_connection(self):
        """Helper to get a connection with pgvector registered"""
        conn = psycopg.connect(self.database_url)
        register_vector(conn)
        return conn

    def initialize_database(self):
        with self._get_connection() as conn:
            with conn.cursor() as cur:
                # enable pgvecto extension 
                cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")

                # create table
                cur.execute(""" 
                    CREATE TABLE IF NOT EXISTS document_chunks (
                        id TEXT PRIMARY KEY,
                        text_content TEXT NOT NULL,
                        embedding VECTOR(3072) NOT NULL,
                        source TEXT,
                        file_type TEXT,
                        page INTEGER,
                        section TEXT,
                        chunk_index INTEGER,
                        parent_document_id TEXT
                    );
                """)

                # # Create HNSW index for faster similarity search
                # cur.execute("""
                #     CREATE INDEX IF NOT EXISTS idx_embedding_hnsw 
                #     ON document_chunks USING hnsw (embedding vector_cosine_ops);
                # """)

                conn.commit()

    # --------------------------------------------------
    # INSERT
    # --------------------------------------------------

    def insert_embeddings(self, embedded_docs: List[Dict]):
        query = """
        INSERT INTO document_chunks (
            id,
            text_content,
            embedding,
            source,
            file_type,
            page,
            section,
            chunk_index,
            parent_document_id
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (id) DO NOTHING;
        """

        with self._get_connection() as conn:
            with conn.cursor() as cur:
                inserted_count = 0
                for doc in embedded_docs:
                    metadata = doc["metadata"]

                    cur.execute(
                        query,
                        (
                            doc["id"],
                            doc["text"],
                            doc["embedding"],
                            metadata.get("source"),
                            metadata.get("file_type"),
                            metadata.get("page"),
                            metadata.get("section"),
                            metadata.get("chunk_index"),
                            metadata.get("parent_document_id"),
                        )
                    )
                    inserted_count += 1

            conn.commit()

    # --------------------------------------------------
    # VECTOR SEARCH
    # --------------------------------------------------

    def similarity_search(self,
                          query_embedding: List[float],
                          k: int = 5, 
                          filters: Optional[Dict[str, Any]] = None
                          ) -> List[tuple]:

        query = """
        SELECT
            id,
            text_content,
            embedding <=> %s::VECTOR AS distance,
            source,
            file_type,
            page,
            section,
            chunk_index,
            parent_document_id
        FROM document_chunks
        """

        where_clauses = []
        query_params = [query_embedding] # for the distance calculation

        # dynamically append filters to WHERE clause
        if filters: 
            allowed_fields = ["source", "file_type", "page", "section", 
                         "chunk_index", "parent_document_id"]
            
            for key, value in filters.items():
                if key in allowed_fields:
                    where_clauses.append(f"{key} = %s")
                    query_params.append(value)

        if where_clauses:
            query += " WHERE " + " AND ".join(where_clauses)

        query += " ORDER BY embedding <=> %s::VECTOR LIMIT %s;"
        query_params.append(query_embedding)  # For ORDER BY
        query_params.append(k)  # For LIMIT

        with self._get_connection() as conn:
            with conn.cursor() as cur:
                cur.execute(query, tuple(query_params))
                rows = cur.fetchall()

        return rows
    
    def get_total_count(self) -> int:
        """Helper method to verify data was inserted"""
        with self._get_connection() as conn:
            with conn.cursor() as cur:
                cur.execute("SELECT COUNT(*) FROM document_chunks")
                return cur.fetchone()[0]

In [122]:
vector_store = VectorStore()
vector_store.initialize_database()
vector_store.insert_embeddings(embedded_docs)
print(f"Total chunks in database: {vector_store.get_total_count()}")

Total chunks in database: 52


In [ ]:
embed = GeminiEmbeddingGenerator()
query_embedding = embed.embed_query(
    "What chip does the MacBook Air use?"
)

In [144]:
query_embedding

[-0.0054470305,
 0.00898754,
 -0.0018865244,
 -0.07967548,
 -0.016977254,
 -0.005119742,
 0.014922488,
 -0.010065046,
 0.0024985066,
 0.028471267,
 -0.021246282,
 -0.007023483,
 0.0131983645,
 -0.015765559,
 0.11514719,
 -0.0104645025,
 0.0043524397,
 -0.010027961,
 0.0038066139,
 -0.023821818,
 0.009709591,
 -0.0086480845,
 0.0072453697,
 -0.009630293,
 -0.010764154,
 -0.017753525,
 0.01919855,
 -0.023903348,
 0.04218385,
 0.010753592,
 0.0323666,
 -0.03281882,
 -0.0079754,
 0.016388835,
 -0.008791038,
 0.01335842,
 0.022901565,
 0.00916073,
 0.023240354,
 -0.01679729,
 0.01455395,
 -0.007485095,
 0.011981265,
 -0.01006909,
 0.005007374,
 0.00019732099,
 -0.01818166,
 -0.009553513,
 -0.04848267,
 0.01265181,
 -0.032201786,
 0.010309084,
 0.013660549,
 -0.17134854,
 -0.008720405,
 0.017186046,
 -0.008814955,
 -0.009142125,
 0.004691762,
 -0.01594699,
 -0.012510146,
 -0.021404032,
 -0.018513374,
 -0.020338997,
 -0.0031837132,
 -0.041564457,
 -0.00069464353,
 0.013614498,
 0.016063958,
 

In [149]:
vector_store = VectorStore()
results = vector_store.similarity_search(
    query_embedding,
    k=10
)

for row in results:
    print(row)

('macbook_air_(13-inch,_m5)_-_tech_specs_page_5_chunk_30', '## **MacBook Air and the Environment**  \nConfigure your MacBook Air at apple.com.  \nM5 with 10-core CPU and 10-core GPU 24GB or 32GB unified memory 1TB, 2TB or 4TB SSD 30W USB-C Power Adapter or 35W Dual USB-C Port Power Adapter or 70W USB-C Power Adapter', 0.27090572194558205, 'MacBook Air (13-inch, M5) - Tech Specs.pdf', 'pdf', 5, '**MacBook Air and the Environment**', 30, 'macbook_air_(13-inch,_m5)_-_tech_specs')
('macbook_air_(13-inch,_m5)_-_tech_specs_page_7_chunk_43', '3. Testing conducted by Apple in January 2026 using pre-production 13-inch MacBook Air systems with Apple M5, 10-core CPU and 8-core GPU, and pre-production 15-inch MacBook Air systems with Apple M5, 10-core CPU and 10-core GPU, all configured with 16GB of unified memory and 512GB SSD. Wireless web battery life tested by browsing 25 popular websites while connected to Wi-Fi. Video streaming battery life tested with 1080p content in Safari while connected

# Reranker 

In [181]:
import os
from typing import List
from dotenv import load_dotenv
import cohere

# from src.retrieval.retriever import RetrievalResult

@dataclass
class RetrievalResult:
    id: str
    text: str
    score: float

    source: str
    file_type: str

    page: Optional[int]
    section: Optional[str]

    chunk_index: Optional[int]
    parent_document_id: Optional[str]

class CohereReranker:
    def __init__(self, model_name: str = "rerank-v3.5"):
        load_dotenv()
        api_key = os.getenv("COHERE_API_KEY")
        if not api_key:
            raise ValueError("COHERE_API_KEY not found in environment variables.")
        
        # Initialize the modern Cohere Client
        self.client = cohere.ClientV2(api_key=api_key)
        self.model_name = model_name

    def rerank(self, query: str, documents: List[RetrievalResult], top_n: int = 5) -> List[RetrievalResult]:
        """
        Reranks a list of candidate RetrievalResults using Cohere's Cross-Encoder API.
        """
        if not documents:
            return []
            
        # Ensure we don't request a top_n larger than our candidate list size
        top_n = min(top_n, len(documents))

        # Cohere expects a clean list of text strings to analyze
        texts_to_rerank = [doc.text for doc in documents]

        # Call Cohere Rerank API
        response = self.client.rerank(
            model=self.model_name,
            query=query,
            documents=texts_to_rerank,
            top_n=top_n
        )

        reranked_results = []
        
        # Map Cohere's response objects back to your structured RetrievalResults
        for result in response.results:
            original_index = result.index
            original_doc = documents[original_index]

            # Update the score with Cohere's precise relevance score
            original_doc.score = float(result.relevance_score)
            reranked_results.append(original_doc)

        return reranked_results

# Retriever

In [186]:
from dataclasses import dataclass
from typing import List, Optional

#from embeddings import GeminiEmbeddingGenerator
#from vector_store import VectorStore


@dataclass
class RetrievalResult:
    id: str
    text: str
    score: float

    source: str
    file_type: str

    page: Optional[int]
    section: Optional[str]

    chunk_index: Optional[int]
    parent_document_id: Optional[str]


class Retriever:

    def __init__(
        self,
        embedder: GeminiEmbeddingGenerator,
        vector_store: VectorStore,
        reranker: Optional[CohereReranker] = None
    ):
        self.embedder = embedder
        self.vector_store = vector_store
        self.reranker = reranker

    def retrieve(
        self,
        query: str,
        k: int = 10,
        filters: Optional[Dict[str, Any]] = None
    ) -> List[RetrievalResult]:

        # Generate query embedding
        query_embedding = self.embedder.embed_query(query)

        # Stage 1: vecto search + metadaa filtering 
        rows = self.vector_store.similarity_search(
            query_embedding=query_embedding,
            k=k, 
            filters=filters
        )

        results = []
        for row in rows:
            (
                chunk_id,
                text_content,
                distance,
                source,
                file_type,
                page,
                section,
                chunk_index,
                parent_document_id,
            ) = row

            results.append(
                RetrievalResult(
                    id=chunk_id,
                    text=text_content,
                    score=1 - float(distance),
                    source=source,
                    file_type=file_type,
                    page=page,
                    section=section,
                    chunk_index=chunk_index,
                    parent_document_id=parent_document_id,
                )
            )

        # Stage 2: Reranking with cohere
        if self.reranker and results: 
            return self.reranker.rerank(query=query, documents=results)

        # fall back if reranker not configured 
        return results

In [187]:
embedder = GeminiEmbeddingGenerator()
vector_store = VectorStore()
reranker = CohereReranker()
retriever = Retriever(embedder=embedder, vector_store=vector_store, reranker=reranker)

retrieved_chunks = retriever.retrieve(
    query="What chip does the MacBook Air use?",
    k=10,)
    #filters={'page': 2})

In [188]:
retrieved_chunks

[RetrievalResult(id='macbook_air_(13-inch,_m5)_-_tech_specs_page_7_chunk_43', text='3. Testing conducted by Apple in January 2026 using pre-production 13-inch MacBook Air systems with Apple M5, 10-core CPU and 8-core GPU, and pre-production 15-inch MacBook Air systems with Apple M5, 10-core CPU and 10-core GPU, all configured with 16GB of unified memory and 512GB SSD. Wireless web battery life tested by browsing 25 popular websites while connected to Wi-Fi. Video streaming battery life tested with 1080p content in Safari while connected to Wi-Fi', score=0.5201313, source='MacBook Air (13-inch, M5) - Tech Specs.pdf', file_type='pdf', page=7, section='**Acoustic Performance**', chunk_index=43, parent_document_id='macbook_air_(13-inch,_m5)_-_tech_specs'),
 RetrievalResult(id='macbook_air_(13-inch,_m5)_-_tech_specs_page_5_chunk_30', text='## **MacBook Air and the Environment**  \nConfigure your MacBook Air at apple.com.  \nM5 with 10-core CPU and 10-core GPU 24GB or 32GB unified memory 1TB

In [158]:
retrieved_chunks

[RetrievalResult(id='macbook_air_(13-inch,_m5)_-_tech_specs_page_5_chunk_30', text='## **MacBook Air and the Environment**  \nConfigure your MacBook Air at apple.com.  \nM5 with 10-core CPU and 10-core GPU 24GB or 32GB unified memory 1TB, 2TB or 4TB SSD 30W USB-C Power Adapter or 35W Dual USB-C Port Power Adapter or 70W USB-C Power Adapter', score=0.729094278054418, source='MacBook Air (13-inch, M5) - Tech Specs.pdf', file_type='pdf', page=5, section='**MacBook Air and the Environment**', chunk_index=30, parent_document_id='macbook_air_(13-inch,_m5)_-_tech_specs'),
 RetrievalResult(id='macbook_air_(13-inch,_m5)_-_tech_specs_page_7_chunk_43', text='3. Testing conducted by Apple in January 2026 using pre-production 13-inch MacBook Air systems with Apple M5, 10-core CPU and 8-core GPU, and pre-production 15-inch MacBook Air systems with Apple M5, 10-core CPU and 10-core GPU, all configured with 16GB of unified memory and 512GB SSD. Wireless web battery life tested by browsing 25 popular w

# Generator.py

In [156]:
from typing import List

from google import genai
from dotenv import load_dotenv

import os

#from retriever import RetrievalResult


class Generator:

    def __init__(
        self,
        model_name: str = "gemini-2.5-flash"
    ):
        load_dotenv()

        api_key = os.getenv("GEMINI_API_KEY")

        if not api_key:
            raise ValueError("GEMINI_API_KEY not found.")

        self.client = genai.Client(api_key=api_key)

        self.model_name = model_name

    def _build_context(
        self,
        retrieved_chunks: List[RetrievalResult]
    ) -> str:

        context_parts = []

        for idx, chunk in enumerate(retrieved_chunks, start=1):

            context_parts.append(
                f"""
[Chunk {idx}]
Source: {chunk.source}
Page: {chunk.page}
Section: {chunk.section}

{chunk.text}
"""
            )

        return "\n\n".join(context_parts)

    def generate_answer(
        self,
        question: str,
        retrieved_chunks: List[RetrievalResult]
    ) -> str:

        context = self._build_context(retrieved_chunks)

        prompt = f"""
You are a helpful assistant answering questions using retrieved document context.

Instructions:
- Answer only using the provided context.
- If the answer is not contained in the context, say:
  "I could not find the answer in the provided documents."
- Be concise and accurate.
- Mention the source i.e id, source, page, section.

Context:
{context}

Question:
{question}
"""

        response = self.client.models.generate_content(
            model=self.model_name,
            contents=prompt
        )

        return response.text

In [189]:
generator = Generator()
answer = generator.generate_answer(
    question = "what chip does the macbook air use?",
    retrieved_chunks = retrieved_chunks
)

In [190]:
answer

'The MacBook Air (13-inch, M5) uses the Apple M5 chip.\n(Source: MacBook Air (13-inch, M5) - Tech Specs.pdf, Page: 1, Section: Apple M5 chip)'

# RAG Pipeline

In [ ]:
from dataclasses import dataclass
from typing import List

#from retriever import Retriever, RetrievalResult
#from generator import Generator

@dataclass
class SourceReference:
    source: str
    page: int | None
    section: str | None

@dataclass
class RAGResponse:
    question: str
    answer: str
    retrieved_chunks: List[RetrievalResult]
    sources: List[SourceReference]


class RAGPipeline:

    def __init__(
        self,
        retriever: Retriever,
        generator: Generator
    ):
        self.retriever = retriever
        self.generator = generator

    def ask(
        self,
        question: str,
        k: int = 10
    ) -> RAGResponse:

        # Step 1: Retrieve
        retrieved_chunks = self.retriever.retrieve(
            query=question,
            k=k
        )

        sources = []

        seen = set()

        for chunk in retrieved_chunks:
        
            key = (
                chunk.source,
                chunk.page,
                chunk.section
            )

            if key not in seen:
                seen.add(key)

                sources.append(
                    SourceReference(
                        source=chunk.source,
                        page=chunk.page,
                        section=chunk.section
                    )
                )

        # Step 2: Generate
        answer = self.generator.generate_answer(
            question=question,
            retrieved_chunks=retrieved_chunks
        )

        return RAGResponse(
            question=question,
            answer=answer,
            retrieved_chunks=retrieved_chunks, 
            sources=sources
        )

In [ ]:
response = rag.ask(
    "What chip does the MacBook Air use?"
)

print(response.answer)